# Clusters zona da barragem

## grid = 100 m x 100 m

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filtrar_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filtrar_area(asc)
desc = filtrar_area(desc)
ortho_v = filtrar_area(ortho_v)
ortho_h = filtrar_area(ortho_h)

# ==============================
# 3. Criar grelha base (100 m)
# ==============================
grid_size = 100
x_edges = np.arange(asc['easting'].min(), asc['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc['northing'].min(), asc['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

# Atribuir células ASC
asc['cell_x'] = pd.cut(asc['easting'], bins=x_edges_shifted, labels=False)
asc['cell_y'] = pd.cut(asc['northing'], bins=y_edges_shifted, labels=False)

# Calcular centroide de cada célula com base nos pontos ASC
agg = asc.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 4. Criar GeoDataFrames
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

points = gpd.GeoDataFrame(
    agg,
    geometry=gpd.points_from_xy(agg['x_center'], agg['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 5. Atribuir células a ORTHO
# ==============================
for ortho_df in [ortho_v, ortho_h]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

# ==============================
# 6. Células selecionadas
# ==============================
selected_ids = [
    "3_4","4_4","5_4", "6_4",
    "4_5","5_5","6_5","7_5",
    "4_6","5_6","6_6","7_6",
    "5_3"
]

agg_sel = agg[agg['cell_id'].isin(selected_ids)]
grid_sel = grid[grid['cell_id'].isin(selected_ids)]
points_sel = points[points['cell_id'].isin(selected_ids)]

ortho_v_sel = ortho_v[ortho_v['cell_id'].isin(selected_ids)]
ortho_h_sel = ortho_h[ortho_h['cell_id'].isin(selected_ids)]

# Combinar ORTHO
ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True)
ortho_comb = ortho_comb.drop_duplicates(subset=['easting', 'northing'])
gdf_ortho_comb = gpd.GeoDataFrame(
    ortho_comb,
    geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 7. Mapa Final
# ==============================
fig, ax = plt.subplots(figsize=(10, 10))

# Grelha base
grid.boundary.plot(ax=ax, color='lightgray', linewidth=0.5, label='Grelha Base (100 m)')

# Células selecionadas
grid_sel.boundary.plot(ax=ax, color='red', linewidth=1.5, label='Células Selecionadas')

# Pontos ASC/DESC
points_sel.plot(ax=ax, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

# Pontos ORTHO
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

# IDs das células
for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax.text(
        x_min, y_max, row['cell_id'],
        fontsize=9, ha='left', va='top', color='black',
        bbox=dict(facecolor='white', alpha=0.6, pad=1)
    )

# Basemap e título
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_title("Mapa das Células Selecionadas com Pontos ASC/DESC e ORTHO", fontsize=14)
ax.set_axis_off()
ax.legend(loc='lower left', fontsize=9, frameon=True)

plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filtrar_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filtrar_area(asc)
desc = filtrar_area(desc)
ortho_v = filtrar_area(ortho_v)
ortho_h = filtrar_area(ortho_h)

# ==============================
# 3. Criar grelha base (100 m)
# ==============================
grid_size = 100
x_edges = np.arange(asc['easting'].min(), asc['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc['northing'].min(), asc['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

# Atribuir células ASC
asc['cell_x'] = pd.cut(asc['easting'], bins=x_edges_shifted, labels=False)
asc['cell_y'] = pd.cut(asc['northing'], bins=y_edges_shifted, labels=False)

# Calcular centroide de cada célula com base nos pontos ASC
agg = asc.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 4. Criar GeoDataFrames
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

points = gpd.GeoDataFrame(
    agg,
    geometry=gpd.points_from_xy(agg['x_center'], agg['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 5. Atribuir células a ORTHO
# ==============================
for ortho_df in [ortho_v, ortho_h]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

# ==============================
# 6. Células selecionadas
# ==============================
selected_ids = [
    "3_4","4_4","5_4", "6_4",
    "4_5","5_5","6_5","7_5",
    "4_6","5_6","6_6","7_6",
    "5_3"
]

agg_sel = agg[agg['cell_id'].isin(selected_ids)]
grid_sel = grid[grid['cell_id'].isin(selected_ids)]
points_sel = points[points['cell_id'].isin(selected_ids)]

ortho_v_sel = ortho_v[ortho_v['cell_id'].isin(selected_ids)]
ortho_h_sel = ortho_h[ortho_h['cell_id'].isin(selected_ids)]

# Combinar ORTHO
ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True)
ortho_comb = ortho_comb.drop_duplicates(subset=['easting', 'northing'])
gdf_ortho_comb = gpd.GeoDataFrame(
    ortho_comb,
    geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 7. Mapa Final
# ==============================
fig, ax = plt.subplots(figsize=(10, 10))

# Grelha base
grid.boundary.plot(ax=ax, color='lightgray', linewidth=0.5, label='Grelha Base (100 m)')

# Células selecionadas
grid_sel.boundary.plot(ax=ax, color='red', linewidth=1.5, label='Células Selecionadas')

# Pontos ASC/DESC
points_sel.plot(ax=ax, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

# Pontos ORTHO
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

# IDs das células
for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax.text(
        x_min, y_max, row['cell_id'],
        fontsize=9, ha='left', va='top', color='black',
        bbox=dict(facecolor='white', alpha=0.6, pad=1)
    )

# Basemap e título
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_title("Mapa das Células Selecionadas com Pontos ASC/DESC e ORTHO", fontsize=14)
ax.set_axis_off()
ax.legend(loc='lower left', fontsize=9, frameon=True)

plt.show()

### Células para clustering

In [ ]:
# ==============================
# FASE 1 - Seleção das células
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np

# ---- 1. Ler CSVs ----
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ---- 2. Filtrar área ----
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filtrar_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filtrar_area(asc)
desc = filtrar_area(desc)
ortho_v = filtrar_area(ortho_v)
ortho_h = filtrar_area(ortho_h)

# ---- 3. Criar grelha base ----
grid_size = 100
x_edges = np.arange(asc['easting'].min(), asc['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc['northing'].min(), asc['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc['cell_x'] = pd.cut(asc['easting'], bins=x_edges_shifted, labels=False)
asc['cell_y'] = pd.cut(asc['northing'], bins=y_edges_shifted, labels=False)

agg = asc.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ---- 4. Criar GeoDataFrames ----
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)
points = gpd.GeoDataFrame(
    agg,
    geometry=gpd.points_from_xy(agg['x_center'], agg['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ---- 5. Selecionar células manualmente ----
selected_ids = [
    "3_4","4_4","5_4","6_4",
    "4_5","5_5","6_5","7_5",
    "4_6","5_6","6_6","7_6",
    "5_3"
]
grid_sel = grid[grid['cell_id'].isin(selected_ids)]
points_sel = points[points['cell_id'].isin(selected_ids)]

# ---- 6. Mapa ----
fig, ax = plt.subplots(figsize=(10,10))
grid.boundary.plot(ax=ax, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax, color='red', linewidth=1.5, label='Células Selecionadas')
points_sel.plot(ax=ax, color='white', edgecolor='black', markersize=60)
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.legend()
ax.set_title("Células Selecionadas para Clustering", fontsize=14)
ax.set_axis_off()
plt.show()

# ---- 7. Guardar lista para Fase 2 ----
pd.DataFrame({'cell_id': selected_ids}).to_csv("selected_cells.csv", index=False)
print("✅ Ficheiro 'selected_cells.csv' criado com sucesso.")


### Temperatura

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme as tuas colunas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Cálculo de β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Grelha e centroides reais
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar grelha GeoDataFrame
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Filtrar células selecionadas
# ==============================
selected_cells = pd.read_csv("selected_cells.csv")['cell_id'].tolist()
agg = agg[agg['cell_id'].isin(selected_cells)]
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)

# ==============================
# 12. Clustering de dV
# ==============================
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 13. Figura: mapa + séries
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ---- Mapa
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

ax_map.legend(fontsize=10, loc='upper left')
ax_map.set_axis_off()
ax_map.set_title(f"Clusters de séries temporais de deslocamento vertical (dV)\n"
                 f"K-Means, K={k}. Grelha {grid_size} m × {grid_size} m", fontsize=15)

# ---- Séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    ax2 = ax.twinx()
    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme as tuas colunas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Cálculo de β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Grelha e centroides reais
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar grelha GeoDataFrame
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Filtrar células selecionadas
# ==============================
selected_cells = pd.read_csv("selected_cells.csv")['cell_id'].tolist()
agg = agg[agg['cell_id'].isin(selected_cells)]
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)

# ==============================
# 12. Clustering de dV
# ==============================
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

from statsmodels.tsa.seasonal import STL

# ==============================
# 13. Figura: mapa + séries + STL
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 20))
gs = fig.add_gridspec(4, n_clusters, height_ratios=[2, 1.2, 1, 1])

# ---- Mapa (linha 1)
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

ax_map.legend(fontsize=10, loc='upper left')
ax_map.set_axis_off()
ax_map.set_title(f"Clusters de séries temporais de deslocamento vertical (dV)\n"
                 f"K-Means, K={k}. Grelha {grid_size} m × {grid_size} m", fontsize=15)

# ---- Séries e STL
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1
temp_min = df_temp['med_smooth'].min()
temp_max = df_temp['med_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    # --- Linha 2: dV médio + temperatura
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)

    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_mean_dV.index, cluster_mean_dV.values, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # Temperatura na escala secundária
    ax2 = ax.twinx()
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    # STL decomposition
    stl = STL(cluster_mean_dV, period=12, robust=True).fit()

    # --- Linha 3: Tendência
    ax_trend = fig.add_subplot(gs[2, idx])
    ax_trend.plot(cluster_mean_dV.index, stl.trend, color=cluster_color, linewidth=2.5)
    ax_trend.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax_trend.set_title(f'Tendência (STL) — Cluster {cluster_id + 1}', fontsize=11)
    ax_trend.grid(True, linestyle="--", alpha=0.4)

    # --- Linha 4: Sazonal + Resíduo
    ax_seas = fig.add_subplot(gs[3, idx])
    ax_seas.plot(cluster_mean_dV.index, stl.seasonal, color='orange', linewidth=2, label='Sazonal')
    ax_seas.plot(cluster_mean_dV.index, stl.resid, color='gray', linewidth=1.5, label='Resíduo')
    ax_seas.legend(fontsize=8, loc='upper right')
    ax_seas.set_title(f'Sazonalidade + Resíduo — Cluster {cluster_id + 1}', fontsize=11)
    ax_seas.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()


### Nível da albufeira

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme as tuas colunas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Cálculo de β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Grelha e centroides reais
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 8b. Criar grelha GeoDataFrame e centroides
# ==============================
from shapely.geometry import box

# Obter pontos médios (centroides das células)
points = asc_interp.groupby('cell_id').agg({
    'easting': 'mean',
    'northing': 'mean'
}).reset_index()

gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# Criar grelha regular de polígonos (usada no mapa)
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(
                x_edges_shifted[ix],
                y_edges_shifted[iy],
                x_edges_shifted[ix+1],
                y_edges_shifted[iy+1]
            )
        })

grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)


# ==============================
# 9. Carregar nível da albufeira
# ==============================
from scipy.signal import savgol_filter

df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])

# Aplica suavização (ajusta 'window' se necessário)
window = 365
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window, polyorder=2)

# ==============================
# 9b. Filtrar apenas células selecionadas
# ==============================
selected_cells = pd.read_csv("selected_cells.csv")

# garantir que a coluna tem o nome 'cell_id'
if 'cell_id' not in selected_cells.columns:
    selected_cells.columns = ['cell_id']

# manter apenas as células selecionadas no conjunto agregado
agg = agg[agg['cell_id'].isin(selected_cells['cell_id'])].copy()

# ==============================
# 10. Clustering de dV (apenas células selecionadas)
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')


# ==============================
# 11. Figura única: mapa + clusters + nível
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa ---
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120, label='Pontos ASC/DESC')
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical (dV)\n"
    f"Correlação com o nível da albufeira.\nK-Means, K={k}.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# --- Linha 2: Séries temporais + nível ---
dV_min, dV_max = agg['dV'].min(), agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1
nivel_min, nivel_max = df_nivel['nivel_smooth'].min(), df_nivel['nivel_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # Série do nível (escala secundária)
    ax2 = ax.twinx()
    nivel_visual = (df_nivel['nivel_smooth'] - nivel_min) / (nivel_max - nivel_min)
    nivel_visual = nivel_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_nivel['data'], nivel_visual, color='black', linewidth=2.2, alpha=0.85, label='Nível da albufeira (m)')
    ax2.set_ylabel("Nível da albufeira (m)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    nivel_ticks_real = np.linspace(nivel_min, nivel_max, 6)
    nivel_ticks_visual = (nivel_ticks_real - nivel_min) / (nivel_max - nivel_min)
    nivel_ticks_visual = nivel_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(nivel_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in nivel_ticks_real])

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


### Precipitação

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha e centroides
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 9. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])

# ==============================
# 9b. Filtrar apenas células selecionadas
# ==============================
selected_cells = pd.read_csv("selected_cells.csv")
if 'cell_id' not in selected_cells.columns:
    selected_cells.columns = ['cell_id']
agg = agg[agg['cell_id'].isin(selected_cells['cell_id'])].copy()

# ==============================
# 10. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 11. Figura única: mapa + séries temporais + precipitação
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: mapa ---
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120, label='Pontos ASC/DESC')
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6, label=f'Cluster {cluster_id + 1} - {n_cells} células')
ax_map.set_title(f"Clusters de séries temporais de dV e precipitação", fontsize=16)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# --- Linha 2: séries temporais ---
dV_min, dV_max = agg['dV'].min(), agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5, label=f'Média Cluster {cluster_id + 1}')
    
    # Precipitação
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='blue', alpha=0.3, label='Precipitação (mm)')
    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec'].max()*1.1)
    
    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')
    ax.grid(False)
    ax2.grid(False)

plt.tight_layout()
plt.show()


## Grid = 50 m x 50 m

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filtrar_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filtrar_area(asc)
desc = filtrar_area(desc)
ortho_v = filtrar_area(ortho_v)
ortho_h = filtrar_area(ortho_h)

# ==============================
# 3. Criar grelha (tamanho ajustável)
# ==============================
grid_size = 50  # <-- muda aqui o tamanho da grelha

x_edges = np.arange(asc['easting'].min(), asc['easting'].max() + grid_size, grid_size)
y_edges = np.arange(asc['northing'].min(), asc['northing'].max() + grid_size, grid_size)
x_edges_shifted = x_edges - grid_size / 2
y_edges_shifted = y_edges - grid_size / 2

grid_data = []
for ix in range(len(x_edges_shifted) - 1):
    for iy in range(len(y_edges_shifted) - 1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix + 1], y_edges_shifted[iy + 1])
        })

grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 4. Calcular centroides por célula (ASC/DESC combinados)
# ==============================
asc["cell_x"] = pd.cut(asc["easting"], bins=x_edges_shifted, labels=False)
asc["cell_y"] = pd.cut(asc["northing"], bins=y_edges_shifted, labels=False)

agg = asc.dropna(subset=["cell_x", "cell_y"]).groupby(["cell_x", "cell_y"]).agg(
    x_center=("easting", "mean"),
    y_center=("northing", "mean")
).reset_index()

agg["cell_id"] = agg["cell_x"].astype(int).astype(str) + "_" + agg["cell_y"].astype(int).astype(str)

points = gpd.GeoDataFrame(
    agg,
    geometry=gpd.points_from_xy(agg["x_center"], agg["y_center"]),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 5. Converter ORTHO para long format + associar células
# ==============================
def melt_ortho(df):
    disp_cols = df.columns[24:]  # ajustar conforme o formato
    long_df = df.melt(id_vars=["easting", "northing"], value_vars=disp_cols,
                      var_name="date", value_name="disp")
    long_df["date"] = pd.to_datetime(long_df["date"], errors="coerce")
    long_df.dropna(subset=["disp", "date"], inplace=True)
    return long_df

ortho_v_long = melt_ortho(ortho_v)
ortho_h_long = melt_ortho(ortho_h)

for ortho_df in [ortho_v_long, ortho_h_long]:
    ortho_df["cell_x"] = pd.cut(ortho_df["easting"], bins=x_edges_shifted, labels=False)
    ortho_df["cell_y"] = pd.cut(ortho_df["northing"], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=["cell_x", "cell_y"], inplace=True)
    ortho_df["cell_id"] = ortho_df["cell_x"].astype(int).astype(str) + "_" + ortho_df["cell_y"].astype(int).astype(str)

selected_ids = agg["cell_id"].unique()
grid_sel = grid[grid["cell_id"].isin(selected_ids)]

# ==============================
# 6. Mapa das células (com contorno vermelho nas com dados)
# ==============================
fig_map, ax_map = plt.subplots(figsize=(10, 10))

# Grelha completa (todas as células, em cinza claro)
grid.boundary.plot(ax=ax_map, color="lightgray", linewidth=0.5, label=f"Grelha Base ({grid_size} m)")

# Células com dados (em vermelho)
grid_sel.boundary.plot(ax=ax_map, color="red", linewidth=1.5, label="Células com pontos ASC/DESC")

# Pontos ASC/DESC (centroides)
points.plot(ax=ax_map, color="white", edgecolor="black", markersize=60, label="Pontos ASC/DESC")

# ORTHO combinados
ortho_comb = pd.concat([ortho_v_long, ortho_h_long], ignore_index=True).drop_duplicates(subset=["easting", "northing", "date"])
gdf_ortho_comb = gpd.GeoDataFrame(
    ortho_comb,
    geometry=gpd.points_from_xy(ortho_comb["easting"], ortho_comb["northing"]),
    crs="EPSG:3035"
).to_crs(epsg=3857)

if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax_map, color="black", markersize=25, alpha=0.8, label="Pontos ORTHO")

# IDs apenas nas células com dados (brancos e no canto superior esquerdo)
for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row["geometry"].bounds
    ax_map.text(
        x_min + (grid_size * 0.1),  # pequeno deslocamento da borda esquerda
        y_max - (grid_size * 0.1),  # pequeno deslocamento da borda superior
        row["cell_id"],
        fontsize=6,
        ha="left", va="top",
        color="white", weight="bold",
        path_effects=[plt.matplotlib.patheffects.withStroke(linewidth=1.5, foreground="black", alpha=0.8)]
    )

# Basemap e layout
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title(f"Mapa das Células com Pontos ASC/DESC e ORTHO ({grid_size} m)", fontsize=14)
ax_map.set_axis_off()
ax_map.legend(loc="lower left", fontsize=9, frameon=True)
plt.show()




In [ ]:
# ==============================
# FASE 1 - Seleção das células
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np

# ---- 1. Ler CSVs ----
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ---- 2. Filtrar área ----
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filtrar_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filtrar_area(asc)
desc = filtrar_area(desc)
ortho_v = filtrar_area(ortho_v)
ortho_h = filtrar_area(ortho_h)

# ---- 3. Criar grelha base ----
grid_size = 50
x_edges = np.arange(asc['easting'].min(), asc['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc['northing'].min(), asc['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc['cell_x'] = pd.cut(asc['easting'], bins=x_edges_shifted, labels=False)
asc['cell_y'] = pd.cut(asc['northing'], bins=y_edges_shifted, labels=False)

agg = asc.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ---- 4. Criar GeoDataFrames ----
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)
points = gpd.GeoDataFrame(
    agg,
    geometry=gpd.points_from_xy(agg['x_center'], agg['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ---- 5. Selecionar células manualmente ----
selected_ids = [
    "6_8", "7_8", "8_7", "6_7",
    "9_7", "7_9", "7_10",
    "8_10", "8_11",
    "9_9", "9_10", "9_11", "9_12",
    "10_8", "10_9", "10_10", "10_11", "10_12",
    "11_8", "11_9", "11_10", "11_11", "11_12",
    #"12_8", "12_9",
    #"12_10",
    "12_11", "12_12",
    #"13_8",
    "13_9", "13_10", "13_11", "13_12",
    #"14_8",
    #"14_9",
    "14_10", "14_11", "14_12",
    #"15_9", "15_10",
    "15_11",
    #"15_12",
    #"16_10",
    #"16_11", "16_12",
    #"17_10", "17_11",
    #"18_10", "18_11",
    #"19_10", "19_11", "19_12",
    #"5_8", "5_9",
    #"4_8", "4_9"
]
grid_sel = grid[grid['cell_id'].isin(selected_ids)]
points_sel = points[points['cell_id'].isin(selected_ids)]

# ---- 6. Mapa ----
fig, ax = plt.subplots(figsize=(10,10))
grid.boundary.plot(ax=ax, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax, color='red', linewidth=1.5, label='Células Selecionadas')
points_sel.plot(ax=ax, color='white', edgecolor='black', markersize=60)
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.legend()
ax.set_title("Células Selecionadas para Clustering", fontsize=14)
ax.set_axis_off()
plt.show()

# ---- 7. Guardar lista para Fase 2 ----
pd.DataFrame({'cell_id': selected_ids}).to_csv("selected_cells.csv", index=False)
print("✅ Ficheiro 'selected_cells.csv' criado com sucesso.")

### Temperatura

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme as tuas colunas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Cálculo de β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Grelha e centroides reais
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar grelha GeoDataFrame
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Filtrar células selecionadas
# ==============================
selected_cells = pd.read_csv("selected_cells.csv")['cell_id'].tolist()
agg = agg[agg['cell_id'].isin(selected_cells)]
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)

# ==============================
# 12. Clustering de dV
# ==============================
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 13. Figura: mapa + séries
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ---- Mapa
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

ax_map.legend(fontsize=10, loc='upper left')
ax_map.set_axis_off()
ax_map.set_title(f"Clusters de séries temporais de deslocamento vertical (dV)\n"
                 f"K-Means, K={k}. Grelha {grid_size} m × {grid_size} m", fontsize=15)

# ---- Séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    ax2 = ax.twinx()
    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme as tuas colunas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Cálculo de β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Grelha e centroides reais
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar grelha GeoDataFrame
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Filtrar células selecionadas
# ==============================
selected_cells = pd.read_csv("selected_cells.csv")['cell_id'].tolist()
agg = agg[agg['cell_id'].isin(selected_cells)]
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)

# ==============================
# 12. Clustering de dV
# ==============================
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

from statsmodels.tsa.seasonal import STL

# ==============================
# 13. Figura: mapa + séries + STL
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 20))
gs = fig.add_gridspec(4, n_clusters, height_ratios=[2, 1.2, 1, 1])

# ---- Mapa (linha 1)
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

ax_map.legend(fontsize=10, loc='upper left')
ax_map.set_axis_off()
ax_map.set_title(f"Clusters de séries temporais de deslocamento vertical (dV)\n"
                 f"K-Means, K={k}. Grelha {grid_size} m × {grid_size} m", fontsize=15)

# ---- Séries e STL
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1
temp_min = df_temp['med_smooth'].min()
temp_max = df_temp['med_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    # --- Linha 2: dV médio + temperatura
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)

    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_mean_dV.index, cluster_mean_dV.values, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # Temperatura na escala secundária
    ax2 = ax.twinx()
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    # STL decomposition
    stl = STL(cluster_mean_dV, period=12, robust=True).fit()

    # --- Linha 3: Tendência
    ax_trend = fig.add_subplot(gs[2, idx])
    ax_trend.plot(cluster_mean_dV.index, stl.trend, color=cluster_color, linewidth=2.5)
    ax_trend.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax_trend.set_title(f'Tendência (STL) — Cluster {cluster_id + 1}', fontsize=11)
    ax_trend.grid(True, linestyle="--", alpha=0.4)

    # --- Linha 4: Sazonal + Resíduo
    ax_seas = fig.add_subplot(gs[3, idx])
    ax_seas.plot(cluster_mean_dV.index, stl.seasonal, color='orange', linewidth=2, label='Sazonal')
    ax_seas.plot(cluster_mean_dV.index, stl.resid, color='gray', linewidth=1.5, label='Resíduo')
    ax_seas.legend(fontsize=8, loc='upper right')
    ax_seas.set_title(f'Sazonalidade + Resíduo — Cluster {cluster_id + 1}', fontsize=11)
    ax_seas.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()


### Nível da albufeira

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme as tuas colunas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Cálculo de β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Grelha e centroides reais
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 8b. Criar grelha GeoDataFrame e centroides
# ==============================
from shapely.geometry import box

# Obter pontos médios (centroides das células)
points = asc_interp.groupby('cell_id').agg({
    'easting': 'mean',
    'northing': 'mean'
}).reset_index()

gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# Criar grelha regular de polígonos (usada no mapa)
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(
                x_edges_shifted[ix],
                y_edges_shifted[iy],
                x_edges_shifted[ix+1],
                y_edges_shifted[iy+1]
            )
        })

grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)


# ==============================
# 9. Carregar nível da albufeira
# ==============================
from scipy.signal import savgol_filter

df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])

# Aplica suavização (ajusta 'window' se necessário)
window = 365
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window, polyorder=2)

# ==============================
# 9b. Filtrar apenas células selecionadas
# ==============================
selected_cells = pd.read_csv("selected_cells.csv")

# garantir que a coluna tem o nome 'cell_id'
if 'cell_id' not in selected_cells.columns:
    selected_cells.columns = ['cell_id']

# manter apenas as células selecionadas no conjunto agregado
agg = agg[agg['cell_id'].isin(selected_cells['cell_id'])].copy()

# ==============================
# 10. Clustering de dV (apenas células selecionadas)
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')


# ==============================
# 11. Figura única: mapa + clusters + nível
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa ---
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120, label='Pontos ASC/DESC')
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical (dV)\n"
    f"Correlação com o nível da albufeira.\nK-Means, K={k}.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# --- Linha 2: Séries temporais + nível ---
dV_min, dV_max = agg['dV'].min(), agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1
nivel_min, nivel_max = df_nivel['nivel_smooth'].min(), df_nivel['nivel_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # Série do nível (escala secundária)
    ax2 = ax.twinx()
    nivel_visual = (df_nivel['nivel_smooth'] - nivel_min) / (nivel_max - nivel_min)
    nivel_visual = nivel_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_nivel['data'], nivel_visual, color='black', linewidth=2.2, alpha=0.85, label='Nível da albufeira (m)')
    ax2.set_ylabel("Nível da albufeira (m)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    nivel_ticks_real = np.linspace(nivel_min, nivel_max, 6)
    nivel_ticks_visual = (nivel_ticks_real - nivel_min) / (nivel_max - nivel_min)
    nivel_ticks_visual = nivel_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(nivel_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in nivel_ticks_real])

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


### Precipitação

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha e centroides
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 9. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])

# ==============================
# 9b. Filtrar apenas células selecionadas
# ==============================
selected_cells = pd.read_csv("selected_cells.csv")
if 'cell_id' not in selected_cells.columns:
    selected_cells.columns = ['cell_id']
agg = agg[agg['cell_id'].isin(selected_cells['cell_id'])].copy()

# ==============================
# 10. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 11. Figura única: mapa + séries temporais + precipitação
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: mapa ---
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120, label='Pontos ASC/DESC')
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6, label=f'Cluster {cluster_id + 1} - {n_cells} células')
ax_map.set_title(f"Clusters de séries temporais de dV e precipitação", fontsize=16)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# --- Linha 2: séries temporais ---
dV_min, dV_max = agg['dV'].min(), agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5, label=f'Média Cluster {cluster_id + 1}')
    
    # Precipitação
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='blue', alpha=0.3, label='Precipitação (mm)')
    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec'].max()*1.1)
    
    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')
    ax.grid(False)
    ax2.grid(False)

plt.tight_layout()
plt.show()
